In [1]:
import pathlib, re, pandas as pd
from tqdm.auto import tqdm
from datetime import datetime

# ── PATHS ────────────────────────────────────────────────────────────────
ROOT      = pathlib.Path().resolve().parents[0]
RAW_DIR   = ROOT / "data" / "raw" / "parcel-data"
CLEAN_DIR = ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV   = CLEAN_DIR / "parcels_jefferson_monthly_full.csv"

In [2]:
# ── JEFFERSON TAX-DIST CODES ─────────────────────────────────────────────
JT_CODES = {"027", "062", "067", "068", "170", "171", "172", "173", "175", "440"}

# ── CANONICAL COLUMN NAMES ──────────────────────────────────────────────
CANON = [
    "snapshot_month",
    "parcel_id", "tax_dist_code",
    "landuse", "proptyp", "pclass",
    "dweltyp", "yearblt", "nostory", "grade",
    "area_a",
    "market_value_land", "market_value_building",
    "aexmtot", "apprtot",
    "tifmlnd", "tifmbld",
]

ALIASES = {
    # parcel ID
    "parid": "parcel_id", "parcelid": "parcel_id",
    "parcelnumber": "parcel_id", "parcel id": "parcel_id",
    # tax district
    "taxdistrict": "tax_dist_code", "tax_district": "tax_dist_code", "taxdist": "tax_dist_code",
    # land use
    "land_use": "landuse", "landusecode": "landuse",
    # property type
    "prop_type": "proptyp", "proptype": "proptyp",
    # square footage
    "totbldgsqft": "area_a", "tot_bldg_sqft": "area_a", "tot_sf": "area_a",
    # market/appraised value
    "aexmlnd": "market_value_land", "aexmbld": "market_value_building",
    "marketvalue_land": "market_value_land",
    "marketvaluebuilding": "market_value_building",
    "aexmtot": "aexmtot", "marketvalue_total": "aexmtot",
    "apprtot": "apprtot", "appr_total": "apprtot",
    # TIF values
    "tifmlnd": "tifmlnd", "tifmbld": "tifmbld",
}

MONTH_RE = re.compile(r"parcel_(\d{2})_(\d{4})\.csv", re.I)

In [3]:
# ── HELPERS ──────────────────────────────────────────────────────────────
def snapshot_from_fname(fname: pathlib.Path) -> pd.Period:
    mm, yyyy = map(int, MONTH_RE.search(fname.name).groups())
    return pd.Period(f"{yyyy}-{mm:02d}", freq="M")

def load_and_clean(fp: pathlib.Path) -> pd.DataFrame:
    df = pd.read_csv(fp, dtype=str, low_memory=False)

    # 1 ▸ normalize header
    df.columns = (df.columns.str.lower()
                             .str.strip()
                             .str.replace(r"\s+", "", regex=True))
    df.rename(columns={c: ALIASES.get(c, c) for c in df.columns}, inplace=True)

    # 2 ▸ ensure parcel_id exists
    if "parcel_id" not in df.columns:
        guess = [c for c in df.columns if re.fullmatch(r"par.*(id|cel)", c)]
        if not guess:
            return pd.DataFrame()
        df.rename(columns={guess[0]: "parcel_id"}, inplace=True)

    # 3 ▸ derive tax_dist_code if missing
    if "tax_dist_code" not in df.columns:
        df["tax_dist_code"] = (
            df["parcel_id"].fillna("")
                          .str.extract(r"^(\d{1,3})")[0]
                          .str.zfill(3)
        )

    # 4 ▸ Jefferson Township slice
    df = df[df["tax_dist_code"].isin(JT_CODES)]
    if df.empty:
        return df

    # 5 ▸ convert numeric columns
    numeric = [
        "area_a", "market_value_land", "market_value_building",
        "aexmtot", "apprtot", "tifmlnd", "tifmbld",
        "yearblt", "nostory"
    ]
    for col in numeric:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 6 ▸ add snapshot_month column
    df.insert(0, "snapshot_month", snapshot_from_fname(fp))

    return df

In [4]:
# ── MAIN ─────────────────────────────────────────────────────────────────
def main():
    files = sorted(RAW_DIR.glob("parcel_*.csv"))
    if not files:
        print("No parcel CSVs found.")
        return

    first = True
    for f in tqdm(files, desc="Processing", unit="file"):
        try:
            chunk = load_and_clean(f)
            if chunk.empty:
                continue
            chunk.to_csv(
                OUT_CSV,
                mode="w" if first else "a",
                header=first,
                index=False
            )
            first = False
        except Exception as ex:
            tqdm.write(f"⚠ Error in {f.name}: {ex}")

    if first:
        print("No Jefferson rows found.")
    else:
        size = OUT_CSV.stat().st_size / 1024**2
        print(f"✓ Full clean file → {OUT_CSV} ({size:,.1f} MB)")

if __name__ == "__main__":
    t0 = datetime.now()
    main()
    print("Finished in", datetime.now() - t0)

Processing:   0%|          | 0/84 [00:00<?, ?file/s]

✓ Full clean file → C:\Repositories\jefferson-township-run-forecasting\data\clean\parcels_jefferson_monthly_full.csv (255.9 MB)
Finished in 0:07:35.752160


In [5]:
in_file = CLEAN_DIR / "parcels_jefferson_monthly_full.csv"
df2 = pd.read_csv(in_file)

In [6]:
for col in df2.columns:
    print(col)

snapshot_month
parcel_id
market_value_land
market_value_building
aexmtot
apprlnd
apprbld
apprtot
audmap
audrtg
landuse
cauv
school
mailad1
mailad2
mailad3
mailad4
trandt
tranyr
name1
name2
name3
owner_add1
owner_add2
nbrhd
flood
pclass
nocards
acrea
price
ann_tax
sthnum
stcont
sthsfx
stdire
stname
stsfx
staddr
usps_city
state
zipcode
descr1
descr2
descr3
taxdesi
valid
area_a
dweltyp
rooms
baths
hbaths
bedrms
aircond
cinbrhd
cond
fireplc
grade
height
nostory
yearblt
proptyp
wall
tifmlnd
tifmbld
point_x
point_y
homstd
bankcode
tax_dist_code


In [7]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549234 entries, 0 to 549233
Data columns (total 69 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   snapshot_month         549234 non-null  object 
 1   parcel_id              549234 non-null  object 
 2   market_value_land      549234 non-null  float64
 3   market_value_building  549234 non-null  float64
 4   aexmtot                549234 non-null  float64
 5   apprlnd                549234 non-null  float64
 6   apprbld                549234 non-null  float64
 7   apprtot                549234 non-null  float64
 8   audmap                 541113 non-null  object 
 9   audrtg                 541113 non-null  float64
 10  landuse                531324 non-null  float64
 11  cauv                   520625 non-null  float64
 12  school                 549234 non-null  int64  
 13  mailad1                536809 non-null  object 
 14  mailad2                127944 non-nu